# Global University
## Ciberseguridad y Desarrollo de Software

<br>

**Materia:** Inteligencia Artificial y Machine Learning

**Equipo:** Hector Oropeza Pelcastre, Sharon Daniela Escobedo Davila, Diego David Lara Martinez

**Profesor:** Jorge Antonio Delgado Magallanes

---

# 02 - CNN Training, Fine-Tuning and Export
## Automatic Fruit Classification using Computer Vision

**Dataset:** [Fruits Classification](https://www.kaggle.com/datasets/utkarshsaxenadn/fruits-classification) - Kaggle

**Classes:** Apple - Banana - Grape - Mango - Strawberry

---
## Notebook Objective

This notebook covers the second stage of the project (**Parcial 2**): balancing/preprocessing justification, training a CNN built from scratch, reporting base metrics and curves, hyperparameter fine-tuning with Optuna, and exporting the final optimized model.

The main objectives of this notebook are:

1. Clone the project repository and install dependencies.
2. Download the real dataset and reproduce the Parcial 1 resize/split pipeline into this project's `data/` layout.
3. Justify the balancing/preprocessing approach (class weights) given the dataset is already balanced.
4. Train the base `FruitCNN` model from scratch and report Accuracy, Precision, Recall, F1-Score, and loss/accuracy curves.
5. Run Optuna hyperparameter search and retrain with the winning configuration.
6. Compare base vs. tuned metrics and export the final model as `.pt`.

**Note:** This notebook is designed to run in Google Colab with a GPU runtime (Runtime > Change runtime type > GPU).

## 1. Clone Repository and Install Dependencies

In [ ]:
!git clone https://github.com/GoldenDiegos/fruits-classifier.git
%cd fruits-classifier/fruit_neural_network_project/project
!pip install -r requirements.txt -q

import sys
from pathlib import Path

# Make sure local packages (models/, training/, evaluation/, ...) are importable
sys.path.insert(0, str(Path.cwd()))
print("Working directory:", Path.cwd())

## 2. Library Imports

> **Note:** If this is a fresh Colab session and you already ran this notebook before, you can skip the Kaggle download in Section 4 by restoring `data/split` from your cached Google Drive zip (Section 3).

In [ ]:
import os
import shutil
import random
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage, Markdown

from PIL import Image
from sklearn.model_selection import train_test_split

print("Libraries imported successfully.")

## 3. Google Drive Mount (Dataset Cache)

Optuna's hyperparameter search runs many training passes, so avoid re-downloading and re-resizing the 10,000 images every Colab session. Mount Drive to cache `data/split` as a zip after the first run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CACHE_DIR = Path("/content/drive/MyDrive/fruits_classifier_parcial2")
DRIVE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_SPLIT_ZIP = DRIVE_CACHE_DIR / "data_split.zip"

print(f"Drive cache path: {DRIVE_SPLIT_ZIP}")
print(f"Cache exists: {DRIVE_SPLIT_ZIP.exists()}")

## 4. Global Configuration

Same configuration values used in Parcial 1's `01_EDA_Preprocessing.ipynb`, pointed at this project's own `data/` folder so everything stays self-contained after `git clone`.

In [ ]:
# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# Image preprocessing
IMAGE_SIZE = (224, 224)

# Dataset split ratios
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

assert round(TRAIN_RATIO + VAL_RATIO + TEST_RATIO, 2) == 1.00, "Split ratios must sum to 1.0"

IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png"]

# Directory paths (inside this project, not the Parcial 1 repo root)
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
SPLIT_DIR = Path("data/split")

print("Configuration loaded successfully.")
print(f"Image size          : {IMAGE_SIZE}")
print(f"Split ratios        : train={TRAIN_RATIO} | val={VAL_RATIO} | test={TEST_RATIO}")
print(f"Random seed         : {RANDOM_SEED}")
print(f"Raw directory       : {RAW_DIR}")
print(f"Processed directory : {PROCESSED_DIR}")
print(f"Split directory     : {SPLIT_DIR}")

## 5. Restore Cached Split (Optional)

If `data_split.zip` already exists on Drive from a previous session, unzip it directly into `data/split` and skip straight to Section 9 (Balancing Justification). Otherwise, continue with the Kaggle download below.

In [ ]:
if DRIVE_SPLIT_ZIP.exists():
    print("Cached split found on Drive. Restoring data/split ...")
    SPLIT_DIR.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(DRIVE_SPLIT_ZIP), str(SPLIT_DIR))
    print("Restored. You can skip ahead to Section 9 if this looks correct.")
else:
    print("No cached split found on Drive yet. Continue with the download/resize/split sections below.")

## 6. Kaggle Setup and Dataset Download

Same process as Parcial 1: upload your `kaggle.json` API token when prompted.

**Note:** Skip this and the following two sections entirely if Section 5 already restored a cached split.

In [ ]:
!pip install kaggle -q

from google.colab import files

print("Upload your kaggle.json file:")
uploaded = files.upload()

if "kaggle.json" not in uploaded:
    raise FileNotFoundError("kaggle.json was not uploaded. Please upload your Kaggle API token file.")

os.makedirs("/root/.config/kaggle", exist_ok=True)
!cp kaggle.json /root/.config/kaggle/kaggle.json
!chmod 600 /root/.config/kaggle/kaggle.json

print("Kaggle credentials configured successfully.")

if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)

for folder in [RAW_DIR, PROCESSED_DIR, SPLIT_DIR / "train", SPLIT_DIR / "val", SPLIT_DIR / "test"]:
    folder.mkdir(parents=True, exist_ok=True)

DATASET_SLUG = "utkarshsaxenadn/fruits-classification"

!kaggle datasets download -d {DATASET_SLUG} -p {str(RAW_DIR)} --unzip

print("Dataset downloaded and extracted successfully.")

## 7. Class Folder Detection

Identical detection logic to Parcial 1's notebook, reused as-is so class names and counts stay consistent with the already-validated EDA.

In [ ]:
candidate_class_dirs = []

for folder in RAW_DIR.rglob("*"):
    if folder.is_dir():
        image_files = [f for f in folder.glob("*") if f.suffix.lower() in IMAGE_EXTENSIONS]
        if len(image_files) > 0:
            candidate_class_dirs.append(folder)

if not candidate_class_dirs:
    raise FileNotFoundError("No image folders were found inside RAW_DIR.")

CLASS_DIRS = sorted(candidate_class_dirs)
UNIQUE_CLASS_NAMES = sorted({class_dir.name for class_dir in CLASS_DIRS})

print("Image folders selected for processing:")
for class_dir in CLASS_DIRS:
    print(f"  - {class_dir.name}")

print(f"\nTotal unique classes detected: {len(UNIQUE_CLASS_NAMES)}")

## 8. Resize to 224x224 and Split (70/15/15)

Same LANCZOS resize and per-class stratified split as Parcial 1, writing into this project's `data/processed` and `data/split`.

In [ ]:
# Resize
if PROCESSED_DIR.exists():
    shutil.rmtree(PROCESSED_DIR)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

resize_errors = []
total_processed = 0

for class_dir in CLASS_DIRS:
    output_class_dir = PROCESSED_DIR / class_dir.name
    output_class_dir.mkdir(parents=True, exist_ok=True)

    image_files = [f for f in class_dir.glob("*") if f.suffix.lower() in IMAGE_EXTENSIONS]

    for img_path in image_files:
        try:
            with Image.open(img_path) as img:
                img = img.convert("RGB")
                img = img.resize(IMAGE_SIZE, Image.LANCZOS)
                img.save(output_class_dir / img_path.name)
            total_processed += 1
        except Exception as e:
            resize_errors.append({"file": str(img_path), "class": class_dir.name, "error": str(e)})

print(f"Images processed successfully : {total_processed}")
print(f"Errors                        : {len(resize_errors)}")

# Split
processed_class_dirs = [d for d in sorted(PROCESSED_DIR.iterdir()) if d.is_dir()]

if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)
for subset in ["train", "val", "test"]:
    (SPLIT_DIR / subset).mkdir(parents=True, exist_ok=True)

for class_dir in processed_class_dirs:
    images = [f for f in class_dir.glob("*") if f.suffix.lower() in IMAGE_EXTENSIONS]

    train_imgs, temp_imgs = train_test_split(
        images, test_size=(1 - TRAIN_RATIO), random_state=RANDOM_SEED, shuffle=True
    )
    val_imgs, test_imgs = train_test_split(
        temp_imgs, test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO), random_state=RANDOM_SEED, shuffle=True
    )

    for subset_name, subset_imgs in {"train": train_imgs, "val": val_imgs, "test": test_imgs}.items():
        destination_dir = SPLIT_DIR / subset_name / class_dir.name
        destination_dir.mkdir(parents=True, exist_ok=True)
        for img_path in subset_imgs:
            shutil.copy2(img_path, destination_dir / img_path.name)

print("Split completed: data/split/{train,val,test}/<ClassName>/")

## 9. Cache the Split to Google Drive

So the next Colab session (needed for Optuna's many trials) can skip Sections 6-8 entirely via Section 5.

In [ ]:
if not DRIVE_SPLIT_ZIP.exists():
    shutil.make_archive(str(DRIVE_SPLIT_ZIP.with_suffix("")), "zip", str(SPLIT_DIR))
    print(f"Cached split saved to: {DRIVE_SPLIT_ZIP}")
else:
    print("Cache already exists on Drive, skipping.")

## 10. Balancing and Preprocessing Justification (Rubric 20%)

Parcial 1's EDA already confirmed the dataset is **perfectly balanced**: 2,000 images per class across all 5 classes, imbalance ratio 1.0x (see `reports/parcial_1_summary.md`). Techniques such as **SMOTE, random undersampling, or oversampling do not apply here** — they exist to correct a numerical imbalance between classes, and there is none to correct.

Instead, the balancing technique implemented in code is **class weighting inside `CrossEntropyLoss`** (`training/losses.py::compute_class_weights`), computed directly from the real training split below. Because the classes are equally represented, the resulting weights should come out close to `1.0` for every class — this is expected and is itself the evidence that the technique is both correctly implemented and correctly diagnosed as unnecessary for *this* dataset, while remaining fully wired into the training pipeline (`main.py` uses it automatically).

In [ ]:
from torchvision import datasets
from training.losses import compute_class_weights

preview_train_dataset = datasets.ImageFolder(root="data/split/train")
weights = compute_class_weights(preview_train_dataset)

for class_name, weight in zip(preview_train_dataset.classes, weights.tolist()):
    print(f"{class_name:<15}: weight = {weight:.4f}")

## 11. Base Model: Train FruitCNN From Scratch (Rubric 30%)

Trains `FruitCNN` (built from scratch: `Conv2d`/`BatchNorm2d`/activation/`MaxPool2d`/`Dropout2d` blocks, no pretrained backbone), evaluates on the held-out test set, and saves metrics + loss/accuracy curves + confusion matrix under `reports/parcial_2/`.

In [ ]:
!python main.py \
    --train-dir data/split/train \
    --val-dir data/split/val \
    --test-dir data/split/test \
    --model-name fruit_cnn \
    --epochs 30 \
    --batch-size 32 \
    --output models/checkpoints/best_model.pt

### Base Model Results

In [ ]:
with open("reports/parcial_2/base_model_metrics.json") as f:
    base_metrics = json.load(f)

print(json.dumps(base_metrics, indent=2))
display(IPImage(filename="reports/parcial_2/base_model_curves.png"))
display(IPImage(filename="reports/parcial_2/base_model_confusion_matrix.png"))

## 12. Fine-Tuning with Optuna (Rubric 30%)

Searches `learning_rate`, `dropout_rate`, `weight_decay`, `activation`, `base_channels`, and `batch_size` over 20 trials (adjust `--n-trials` up if your Colab GPU quota allows more), then retrains the winning configuration for the full epoch budget and exports it.

In [ ]:
!python scripts/tune.py --n-trials 20 --search-epochs 12 --final-epochs 30

### Base vs. Tuned Comparison

In [ ]:
display(Markdown(Path("reports/parcial_2/comparison_table.md").read_text()))
display(IPImage(filename="reports/parcial_2/tuned_model_curves.png"))
display(IPImage(filename="reports/parcial_2/tuned_model_confusion_matrix.png"))

## 13. Export Final Model (Rubric 20%)

`models/checkpoints/tuned_model.pt` already contains the state dict plus metadata (class names, winning hyperparameters, Optuna best value, final test metrics) via `Trainer.save_checkpoint`. Since `.pt` files are gitignored, copy it to Drive so it survives the Colab runtime being recycled.

In [ ]:
tuned_checkpoint_path = Path("models/checkpoints/tuned_model.pt")
drive_model_path = DRIVE_CACHE_DIR / "tuned_model.pt"

shutil.copy(tuned_checkpoint_path, drive_model_path)
print(f"Copied final model to: {drive_model_path}")

import torch
checkpoint = torch.load(tuned_checkpoint_path, map_location="cpu")
print("Checkpoint metadata:")
print(json.dumps(checkpoint["metadata"], indent=2, default=str))

## 14. Conclusions

- The dataset was confirmed balanced in Parcial 1 (1.0x imbalance ratio); class weighting was implemented and wired into training as the required balancing technique, with near-uniform weights confirming no correction was numerically needed.
- `FruitCNN`, built entirely from scratch (no pretrained backbone), was trained on the real 10,000-image split and evaluated on the held-out test set.
- Optuna hyperparameter search (learning rate, dropout, weight decay, activation, base channels, batch size) produced a tuned configuration that is compared against the base model in `reports/parcial_2/comparison_table.md`.
- The final tuned model was exported as `models/checkpoints/tuned_model.pt`, including metadata with the winning hyperparameters and final test metrics.